# Pronóstico 2027 de la abstención por municipio y semáforo

**Objetivo.** Pronosticar la abstención de cada **municipio** en la elección federal de 2027 (**sin intervención**), asignarle un color de semáforo (verde, amarillo o rojo) y una **confianza** que dice qué tan respaldado está el pronóstico. El modelo principal es **gradient boosting**, que ganó en `notebooks/pruebas/comparacion_modelos.ipynb`; la **regresión logística** se calcula junto a él como comparación sencilla de explicar.

**Datos.** Solo se lee `Datos/datos_procesados/base_tratada.csv`: un registro por grupo (municipio × sexo × rango de edad × tipo de sección) en cada elección. Las variables externas son las columnas de 0 y 1 (one-hot) más la **abstención del mismo grupo en la elección anterior** (`ANT1`). No hay datos externos.

**Diferencia con la comparación de modelos.** Allí solo se usaron los 2,380 municipios con las 6 elecciones, porque las series de tiempo lo exigían. Gradient boosting solo necesita la elección anterior, así que aquí se usan **todos los municipios que tengan 2024**. Para saber si eso cambia el error, la sección 5 repite la validación cruzada con todos los municipios y separa los que tienen las 6 elecciones de los incompletos.

**Qué se hace, en orden.**
1. Se calcula `ANT1` para cada grupo (la abstención de la elección anterior de ese grupo).
2. Se valida el modelo con 3 pruebas (predecir 2018, 2021 y 2024 entrenando solo con lo anterior).
3. Se entrena con todo el historial y se pronostica 2027: cada grupo de 2024 pasa a ser un grupo de 2027 de elección intermedia, con su abstención de 2024 como historia.
4. Los pronósticos de los grupos se juntan por municipio, se asigna el semáforo y la confianza y se guarda `Datos/datos_procesados/pronostico_2027.csv`.

**Supuestos importantes** (se repiten al final).
- La composición de cada municipio en 2027 (cuántas personas hay en cada grupo) es la de 2024.
- 2027 es una elección **intermedia**. El modelo solo ha visto dos intermedias con historia (2015 y 2021), y 2015 fue atípica.

## 1. Configuración y lectura
Se define la ruta de `base_tratada.csv`, dónde se guardará el pronóstico, las 6 elecciones y el año siguiente de cada una (`SIGUIENTE`, se usa para pegar el historial), y se lee la base.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import statsmodels.api as sm
from sklearn.ensemble import HistGradientBoostingRegressor

warnings.filterwarnings("ignore")

_cwd = Path.cwd().resolve()
RAIZ = next(p for p in [_cwd, *_cwd.parents] if (p / "Datos").is_dir())
RUTA_BASE = RAIZ / "Datos" / "datos_procesados" / "base_tratada.csv"
RUTA_SALIDA = RAIZ / "Datos" / "datos_procesados" / "pronostico_2027.csv"

ANIOS = [2009, 2012, 2015, 2018, 2021, 2024]
SIGUIENTE = {ANIOS[i]: ANIOS[i + 1] for i in range(len(ANIOS) - 1)}
PRUEBAS = [2018, 2021, 2024]

base = pd.read_csv(RUTA_BASE, encoding="utf-8-sig")
print(f"{len(base):,} filas | {base.shape[1]} columnas")
base.head()

## 2. Variables y comprobación de la base
- `MUNICIPIO`: un municipio se identifica por estado y nombre (hay nombres repetidos entre estados).
- `GRUPO`: municipio + las columnas de sexo, edad y tipo de sección. Un grupo es, por ejemplo, "mujeres de 30-39 años en zona rural de ese municipio".
- `ONEHOT`: las variables externas del modelo. De cada variable se quita la primera categoría (la de referencia: intermedia, hombre, 18-29 años, sección mixta), porque con todas las columnas sumarían siempre 1 y repetirían información.

Se comprueba que cada grupo aparezca una sola vez por elección (si no, el historial se mezclaría).

In [ ]:
MUNICIPIO = ["ESTADO", "MUNICIPIO"]
COLUMNAS_GRUPO = [c for c in base.columns if c.startswith(("SEXO_", "EDAD_", "SEC_"))]
GRUPO = MUNICIPIO + COLUMNAS_GRUPO

ONEHOT = (["TIPO_Presidencial", "SEXO_Mujer"]
          + [c for c in base.columns if c.startswith("EDAD_")][1:]
          + [c for c in base.columns if c.startswith("SEC_")][1:])

print(f"{len(ONEHOT)} variables one-hot:")
print(ONEHOT)
print("Grupos repetidos en una misma elección:", base.duplicated(GRUPO + ["AELEC"]).sum())
print(f"Municipios: {base.groupby(MUNICIPIO).ngroups:,}")

## 3. Historial de cada grupo (`ANT1`)
`ANT1` es la abstención del **mismo grupo** (mismo municipio, sexo, edad y sección) en la elección anterior. `ANT1_PRES` es `ANT1` multiplicada por `TIPO_Presidencial`: deja que la relación con la elección anterior sea distinta en presidenciales y en intermedias.

El historial se pega **por año** (2009 → 2012, 2012 → 2015, …) y no con un simple corrimiento de filas, porque ahora hay municipios y grupos que no están en todas las elecciones y un corrimiento tomaría el año equivocado.

Si un grupo no existía en la elección anterior (pero el municipio sí), no tiene historial propio. En ese caso se le asigna la abstención que tuvo **todo el municipio** en la elección anterior (se suma primero y se divide después). Esos grupos no se usan para entrenar; solo reciben pronóstico.

Tabla de apoyo `serie_mun`: abstención real de cada municipio en cada elección, sumando primero `NV` y `SV` y dividiendo después.

In [ ]:
serie_mun = base.groupby(MUNICIPIO + ["AELEC"])[["SV", "NV"]].sum()
serie_mun["PESO"] = serie_mun["SV"] + serie_mun["NV"]
serie_mun["Real"] = serie_mun["NV"] / serie_mun["PESO"]

# Abstención del grupo en la elección anterior
anterior = base[GRUPO + ["AELEC", "ABSTENCION"]].copy()
anterior["AELEC"] = anterior["AELEC"].map(SIGUIENTE)
anterior = anterior.dropna(subset=["AELEC"]).astype({"AELEC": int}).rename(columns={"ABSTENCION": "ANT1"})

# Abstención del municipio en la elección anterior (respaldo)
anterior_mun = serie_mun["Real"].reset_index()
anterior_mun["AELEC"] = anterior_mun["AELEC"].map(SIGUIENTE)
anterior_mun = anterior_mun.dropna(subset=["AELEC"]).astype({"AELEC": int}).rename(columns={"Real": "ANT1_MUN"})

tabla = base.merge(anterior, on=GRUPO + ["AELEC"], how="left").merge(anterior_mun, on=MUNICIPIO + ["AELEC"], how="left")
tabla["TIENE_ANT1"] = tabla["ANT1"].notna()
tabla["ANT1"] = tabla["ANT1"].fillna(tabla["ANT1_MUN"])
tabla["ANT1_PRES"] = tabla["ANT1"] * tabla["TIPO_Presidencial"]
COLUMNAS = ONEHOT + ["ANT1", "ANT1_PRES"]

print(f"{len(tabla):,} filas")
print(f"Con historial propio del grupo: {tabla['TIENE_ANT1'].sum():,}")
print(f"Con historial del municipio (grupo nuevo): {(~tabla['TIENE_ANT1'] & tabla['ANT1'].notna()).sum():,}")
print(f"Sin historial (2009 u otro hueco): {tabla['ANT1'].isna().sum():,}")
tabla[["AELEC", "ABSTENCION", "ANT1", "ANT1_PRES", "TIENE_ANT1"]].describe().round(3)

## 4. Modelos, semáforo y métricas
- `peso_de`: cada fila pesa lo que pesa su grupo (`SV + NV`, dividido entre su promedio), para que un grupo de 5 personas no cuente igual que uno de 5,000.
- `boosting`: gradient boosting (200 árboles pequeños de profundidad 4, cada uno corrige los errores del anterior). Es el modelo principal.
- `logistica`: regresión logística binomial (modela `NV` contra `SV` de cada grupo, así que nunca da valores fuera de 0 a 100 %). Es la comparación sencilla de explicar.
- `juntar`: convierte el pronóstico de cada grupo en pronóstico del municipio, con un promedio ponderado por `SV + NV`. Cada pronóstico se recorta a [0, 1].
- `tercil`: da 1, 2 o 3 según el tercio en que cae un valor (1 = menor abstención). Es el semáforo, provisional: verde, amarillo y rojo por posición entre los municipios.
- `metricas`: compara pronóstico y real en un conjunto de municipios. RMSE y MAE son el error típico en puntos de abstención; el sesgo es el error promedio con signo (positivo = el modelo predice más abstención de la real); el acierto del semáforo es el % de municipios cuyo color pronosticado coincide con el real (al azar sería 33 %).

In [ ]:
def peso_de(tabla):
    p = tabla["SV"] + tabla["NV"]
    return p / p.mean()

def boosting(train, test):
    m = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.1, max_depth=4, random_state=0)
    m.fit(train[COLUMNAS], train["ABSTENCION"], sample_weight=peso_de(train))
    return m.predict(test[COLUMNAS])

def logistica(train, test):
    X = sm.add_constant(train[COLUMNAS], has_constant="add")
    y = np.column_stack([train["NV"], train["SV"]])
    m = sm.GLM(y, X, family=sm.families.Binomial()).fit()
    return m.predict(sm.add_constant(test[COLUMNAS], has_constant="add"))

MODELOS = {"Gradient boosting": boosting, "Regresión logística": logistica}

def juntar(test, pron):
    t = test[MUNICIPIO].copy()
    t["PESO"] = test["SV"] + test["NV"]
    t["PRON_PESO"] = np.clip(pron, 0, 1) * t["PESO"]
    m = t.groupby(MUNICIPIO)[["PESO", "PRON_PESO"]].sum()
    return (m["PRON_PESO"] / m["PESO"]).rename("Pronóstico")

def tercil(valores):
    return np.ceil(valores.rank(pct=True) * 3)

def metricas(tabla):
    error = tabla["Pronóstico"] - tabla["Real"]
    return pd.Series({
        "RMSE": np.sqrt((error ** 2).mean()) * 100,
        "MAE": error.abs().mean() * 100,
        "Sesgo": error.mean() * 100,
        "Acierto semáforo (%)": (tercil(tabla["Real"]) == tercil(tabla["Pronóstico"])).mean() * 100,
        "Municipios": len(tabla),
    })

## 5. Validación cruzada con todos los municipios
Para cada año de prueba (2018, 2021, 2024) se entrena con las elecciones anteriores (solo grupos con historial propio) y se pronostican los grupos de ese año, de **todos** los municipios que tengan datos. Las métricas se calculan por prueba y se promedian.

Los municipios se separan en dos tipos para ver si el modelo funciona igual en los incompletos:
- **Completo:** tiene las 6 elecciones.
- **Incompleto:** le falta alguna.

La abstención del año que se predice solo se usa para medir el error, nunca para predecir. Los pronósticos de grupos que no tienen historial propio usan la abstención del municipio como `ANT1`.

In [ ]:
elecciones_mun = base.groupby(MUNICIPIO)["AELEC"].nunique().rename("ELECCIONES")
real = serie_mun["Real"].reset_index().rename(columns={"AELEC": "Año", "Real": "Real"})

bloques = []
for anio in PRUEBAS:
    train = tabla[(tabla["AELEC"] < anio) & tabla["TIENE_ANT1"]]
    test = tabla[(tabla["AELEC"] == anio) & tabla["ANT1"].notna()]
    for nombre, funcion in MODELOS.items():
        pron = juntar(test, funcion(train, test)).reset_index()
        pron["Modelo"], pron["Año"] = nombre, anio
        bloques.append(pron)
        print(f"Prueba {anio}: {nombre} ({len(train):,} filas de entrenamiento, {len(test):,} de prueba)")

validacion = pd.concat(bloques, ignore_index=True)
validacion = validacion.merge(real, on=MUNICIPIO + ["Año"], how="left").merge(elecciones_mun.reset_index(), on=MUNICIPIO, how="left")
validacion["Tipo"] = np.where(validacion["ELECCIONES"] == len(ANIOS), "Completo", "Incompleto")
validacion.head()

### Métricas por prueba y promedio
Cada fila es un modelo, un año de prueba y un tipo de municipio. `Todos` junta completos e incompletos (el semáforo se calcula sobre todos los municipios de esa prueba). Después se promedian las 3 pruebas.

In [ ]:
filas = []
for (modelo, anio), tabla_p in validacion.groupby(["Modelo", "Año"], sort=False):
    filas.append(metricas(tabla_p).rename((modelo, anio, "Todos")))
    for tipo, tabla_t in tabla_p.groupby("Tipo"):
        filas.append(metricas(tabla_t).rename((modelo, anio, tipo)))
por_prueba = pd.DataFrame(filas)
por_prueba.index.names = ["Modelo", "Año", "Tipo"]
por_prueba.round(2)

In [ ]:
promedio = por_prueba.groupby(["Modelo", "Tipo"]).mean().drop(columns="Municipios")
promedio["Municipios (prom.)"] = por_prueba.groupby(["Modelo", "Tipo"])["Municipios"].mean()
print("Referencia de comparacion_modelos.ipynb (solo 2,380 completos): boosting RMSE 9.44, acierto 62.5 %")
promedio.round(2)

### Qué tan bien acierta el color del semáforo
Para el gradient boosting se cruza el color **real** de cada municipio con el color **pronosticado** (en cada prueba, sobre todos los municipios) y se promedian las 3 pruebas. Cada fila suma 100 %: de los municipios que eran verdes, qué % quedó verde, amarillo o rojo en el pronóstico. La diagonal son los aciertos; el error más costoso es la esquina de abajo a la izquierda (rojos reales marcados como verdes) y la de arriba a la derecha.

In [ ]:
COLORES = {1: "Verde", 2: "Amarillo", 3: "Rojo"}
PINTURA = {1: "#2e9e4f", 2: "#f2c230", 3: "#d64541"}

matrices = []
for anio, tabla_p in validacion[validacion["Modelo"] == "Gradient boosting"].groupby("Año"):
    cruce = pd.crosstab(tercil(tabla_p["Real"]), tercil(tabla_p["Pronóstico"]), normalize="index") * 100
    matrices.append(cruce.reindex(index=[1, 2, 3], columns=[1, 2, 3], fill_value=0))
confusion = sum(matrices) / len(matrices)

fig, eje = plt.subplots(figsize=(6, 5))
for i in (1, 2, 3):
    for j in (1, 2, 3):
        valor = confusion.loc[i, j]
        eje.add_patch(Rectangle((j - 1, 3 - i), 1, 1, facecolor=PINTURA[j], alpha=0.15 + 0.85 * valor / 100, edgecolor="white"))
        eje.text(j - 0.5, 3.5 - i, f"{valor:.0f} %", ha="center", va="center", fontsize=14)
eje.set_xlim(0, 3)
eje.set_ylim(0, 3)
eje.set_xticks([0.5, 1.5, 2.5], [COLORES[j] for j in (1, 2, 3)])
eje.set_yticks([0.5, 1.5, 2.5], [COLORES[i] for i in (3, 2, 1)])
eje.set_xlabel("Color pronosticado")
eje.set_ylabel("Color real")
eje.set_title("Gradient boosting: color real vs. pronosticado (promedio de 3 pruebas)")
plt.tight_layout()
plt.show()

## 6. Pronóstico 2027
**Entrenamiento:** todas las filas con historial propio del grupo (elecciones 2012 a 2024), de todos los municipios.

**Datos de 2027:** cada grupo que existe en 2024 se convierte en un grupo de 2027. Es una elección **intermedia** (`TIPO_Presidencial = 0`, por lo tanto `ANT1_PRES = 0`) y su `ANT1` es la abstención del grupo en 2024. Como todavía no se conoce el tamaño de los grupos en 2027, se usan las personas de 2024 como pesos para juntar los grupos en municipio.

Los municipios que no tienen datos en 2024 no tienen historial reciente y **no reciben pronóstico**.

In [ ]:
entrenamiento = tabla[tabla["TIENE_ANT1"]]
ultima = tabla[tabla["AELEC"] == 2024]

datos_2027 = ultima[GRUPO + ["SV", "NV"]].copy()
datos_2027["TIPO_Presidencial"] = 0
datos_2027["ANT1"] = ultima["ABSTENCION"].to_numpy()
datos_2027["ANT1_PRES"] = 0.0

print(f"Entrenamiento: {len(entrenamiento):,} filas | grupos de 2027: {len(datos_2027):,} en {datos_2027.groupby(MUNICIPIO).ngroups:,} municipios")
pronosticos = {}
for nombre, funcion in MODELOS.items():
    pronosticos[nombre] = juntar(datos_2027, funcion(entrenamiento, datos_2027))
    print("Listo:", nombre)

## 7. Tabla por municipio: pronóstico, semáforo y confianza
- `ELECCIONES`: en cuántas de las 6 elecciones aparece el municipio.
- `ABST_2024`: abstención real de 2024 del municipio. `PRON_2027_BOOSTING` y `PRON_2027_LOGISTICA`: pronósticos sin intervención.
- **Semáforo:** terciles del pronóstico del gradient boosting entre los municipios con pronóstico: tercio con menor abstención = verde, tercio de en medio = amarillo, tercio con mayor abstención = rojo. Es una definición provisional. Los municipios sin pronóstico quedan como `Sin datos suficientes`.
- **Confianza:**
  - Alta: tiene las 6 elecciones.
  - Media: tiene 5 elecciones y datos de 2024.
  - Baja: tiene de 1 a 4 elecciones y datos de 2024 (incluye municipios nuevos, de los que el modelo conoce una sola elección).
  - Sin pronóstico: no tiene datos de 2024.

In [ ]:
municipios = elecciones_mun.to_frame()
datos_2024 = serie_mun.xs(2024, level="AELEC")[["PESO", "Real"]].rename(columns={"PESO": "PERSONAS_2024", "Real": "ABST_2024"})
municipios = (municipios.join(datos_2024)
                        .join(pronosticos["Gradient boosting"].rename("PRON_2027_BOOSTING"))
                        .join(pronosticos["Regresión logística"].rename("PRON_2027_LOGISTICA")))

def confianza(fila):
    if pd.isna(fila["PRON_2027_BOOSTING"]):
        return "Sin pronóstico"
    if fila["ELECCIONES"] == 6:
        return "Alta"
    return "Media" if fila["ELECCIONES"] == 5 else "Baja"

municipios["CONFIANZA"] = municipios.apply(confianza, axis=1)
municipios["SEMAFORO"] = tercil(municipios["PRON_2027_BOOSTING"]).map(COLORES).fillna("Sin datos suficientes")
municipios["CAMBIO_VS_2024"] = municipios["PRON_2027_BOOSTING"] - municipios["ABST_2024"]
municipios = municipios.reset_index()

print(f"Municipios: {len(municipios):,}")
pd.crosstab(municipios["SEMAFORO"], municipios["CONFIANZA"], margins=True)

In [ ]:
orden_semaforo = ["Verde", "Amarillo", "Rojo", "Sin datos suficientes"]
resumen = municipios.groupby("SEMAFORO").agg(Municipios=("MUNICIPIO", "size"),
                                              Personas_2024=("PERSONAS_2024", "sum"),
                                              Pronostico_promedio=("PRON_2027_BOOSTING", "mean")).reindex(orden_semaforo)
resumen["Pronostico_promedio"] = (resumen["Pronostico_promedio"] * 100).round(1)
resumen

### Revisión de razonabilidad (no es parte del modelo)
Solo se compara el nivel de los pronósticos con lo observado en las elecciones anteriores, para detectar si el modelo da algo absurdo. La abstención de cada elección se calcula sumando las personas de todos los municipios con datos en ambas elecciones y dividiendo después. Recuerda que en la validación el modelo **subestimó la abstención de 2024**, así que el nivel general es el supuesto más incierto.

In [ ]:
con_pron = municipios.dropna(subset=["PRON_2027_BOOSTING"])
pesos = con_pron["PERSONAS_2024"]
print(f"Pronóstico 2027 (boosting):  {np.average(con_pron['PRON_2027_BOOSTING'], weights=pesos):.1%}")
print(f"Pronóstico 2027 (logística): {np.average(con_pron['PRON_2027_LOGISTICA'], weights=pesos):.1%}")
for anio in (2021, 2024):
    reales = serie_mun.xs(anio, level="AELEC").reindex(pd.MultiIndex.from_frame(con_pron[MUNICIPIO]))
    print(f"Real {anio}: {reales['NV'].sum() / reales['PESO'].sum():.1%} (mismos municipios)")
print(f"Pronósticos fuera de [0, 1]: {((con_pron['PRON_2027_BOOSTING'] < 0) | (con_pron['PRON_2027_BOOSTING'] > 1)).sum()}")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))
colores_grafica = {"Verde": PINTURA[1], "Amarillo": PINTURA[2], "Rojo": PINTURA[3], "Sin datos suficientes": "tab:gray"}

cuenta = pd.crosstab(municipios["CONFIANZA"], municipios["SEMAFORO"]).reindex(index=["Alta", "Media", "Baja", "Sin pronóstico"], columns=orden_semaforo, fill_value=0)
cuenta.plot(kind="bar", stacked=True, ax=ejes[0], color=[colores_grafica[c] for c in cuenta.columns])
ejes[0].set_title("Municipios por confianza y semáforo")
ejes[0].set_xlabel("")
ejes[0].tick_params(axis="x", rotation=0)

ejes[1].hist(con_pron["PRON_2027_BOOSTING"] * 100, bins=40, color="tab:blue")
ejes[1].set_title("Pronóstico 2027 de abstención por municipio (%)")
ejes[1].set_xlabel("Abstención pronosticada (%)")
plt.tight_layout()
plt.show()

### Municipios sin pronóstico o con baja confianza
Estos municipios hay que mencionarlos en el reporte: los que no tienen datos de 2024 quedan sin pronóstico, y los de baja confianza tienen una sola elección o muy pocas como historia.

In [ ]:
columnas_vista = ["ESTADO", "MUNICIPIO", "ELECCIONES", "PERSONAS_2024", "ABST_2024", "PRON_2027_BOOSTING", "SEMAFORO", "CONFIANZA"]
print("Sin pronóstico:")
display(municipios[municipios["CONFIANZA"] == "Sin pronóstico"][columnas_vista[:3] + ["SEMAFORO"]].reset_index(drop=True))
print("Baja confianza (los 15 más grandes):")
display(municipios[municipios["CONFIANZA"] == "Baja"].sort_values("PERSONAS_2024", ascending=False)[columnas_vista].head(15).round(3).reset_index(drop=True))

### Los 15 municipios en rojo con más personas
Ordenados por tamaño: son los rojos donde más personas podrían dejar de votar, un primer insumo para priorizar recursos.

In [ ]:
rojos = municipios[municipios["SEMAFORO"] == "Rojo"].copy()
rojos["PERSONAS_QUE_NO_VOTARIAN"] = (rojos["PERSONAS_2024"] * rojos["PRON_2027_BOOSTING"]).round(0)
rojos.sort_values("PERSONAS_QUE_NO_VOTARIAN", ascending=False)[columnas_vista + ["PERSONAS_QUE_NO_VOTARIAN"]].head(15).round(3).reset_index(drop=True)

## 8. Guardar el pronóstico
Se guarda la tabla por municipio en `Datos/datos_procesados/pronostico_2027.csv` (`utf-8-sig`, para que Excel muestre bien los acentos).

In [ ]:
columnas_salida = ["ESTADO", "MUNICIPIO", "ELECCIONES", "PERSONAS_2024", "ABST_2024",
                   "PRON_2027_BOOSTING", "PRON_2027_LOGISTICA", "CAMBIO_VS_2024", "SEMAFORO", "CONFIANZA"]
municipios[columnas_salida].round(4).to_csv(RUTA_SALIDA, index=False, encoding="utf-8-sig")
print(f"Guardado: {RUTA_SALIDA.name} ({len(municipios):,} municipios)")

## 9. Supuestos y límites (para el reporte)
- **Elección intermedia:** el modelo conoce solo dos intermedias con historial (2015, que fue atípica, y 2021). El nivel general de abstención de 2027 es el supuesto más incierto; en la validación el modelo subestimó la abstención de 2024. El **orden** de los municipios es más confiable que el valor exacto.
- **Composición de 2024:** se supone que en 2027 cada municipio tiene los mismos grupos y tamaños que en 2024.
- **Semáforo por posición:** los colores son terciles entre los municipios (provisional); indican qué municipios están en riesgo relativo, no un umbral absoluto. Es más confiable en los extremos que en el grupo amarillo.
- **Municipios con poca historia:** los de baja confianza dependen de una sola elección; los 12 sin datos de 2024 no tienen pronóstico.
- **Sin intervención:** este pronóstico es el escenario base. El escenario con intervención hipotética se construye aparte, a partir de este pronóstico.